# 52 Timflow model: calibration, analysis and storm event

<!-- navigation -->
[<kbd>← 51 Prepare model inputs</kbd>](51%20Prepare%20model%20inputs.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>60 Summary over transects →</kbd>](60%20Summary%20over%20transects.ipynb)

**Required** (the storm event part is optional) · minimal path: 00 → 01 → 10 → 20 → 51 → **52**

Builds the steady and transient [Timflow](https://github.com/timflow-org/timflow) models of one transect ([Timflow documentation](https://timflow.readthedocs.io)), calibrates them, computes leakage factors and response factors, and the heads during the design hydrograph (storm event).

| | |
|---|---|
| **Input** | `observations_model_period.xlsx` (notebook 51), `model_parameters_<transect>.xlsx`, transects metadata, piezometers, storm event file (optional) |
| **Output** | calibrated parameters (new tab in the parameter file), figures and response factors in the transect result folders |

In [ ]:
from pathlib import Path

import timflow_levee_toolkit as tlt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import timflow as tf

tlt.show_versions()
tf.show_versions()

In [ ]:
## UPDATE FOR YOUR PROJECT: the project folder (with project_settings.toml and the data)
project_dir = Path.cwd().parent / "examples" / "papendrecht"

project = tlt.load_project(project_dir)
units = project.units
project

In [ ]:
## UPDATE FOR YOUR PROJECT: the transect to work on (a name from the transects metadata)
transect_name = "Papendrecht West"

## 1. Inputs
Metadata, piezometers and the observations of the model period (from notebook 51).

In [ ]:
location_name = f"{project.name} - {transect_name}"

meta = project.transect_metadata(transect_name)
loc_river = meta["loc_river"]
loc_storm_event = meta["loc_storm_event"]
loc_farfield = meta["farfield"]
L_river = meta["L_river"]
L_foreland = meta["L_foreland"]
L_hinterland = meta["L_hinterland"]  # only for plots: in the model the hinterland extends to infinity

obslocs = tlt.make_obslocs(project.piezometers(transect_name))

# Observations of the model period from notebook 51, with the time in days since its start
obs_ref = tlt.read_observations(project.transect_dir(transect_name, "timeseries") / "observations_model_period.xlsx")
obs_dh = obs_ref - obs_ref.mean()

savefolder = project.transect_dir(transect_name, "intermediate")
savefolder_final = project.transect_dir(transect_name, "final")

obs_summary = tlt.summarize_observations(obs_ref, obslocs, loc_farfield, loc_river)
obs_avg_farfieldref = obs_summary["h_avg_farfield"]

Calculate the `tsandhstar` boundary condition at the river for the daily (**tidal**) situation: tidal fluctuation at the river minus the average river level.

_See the [Timflow documentation](https://timflow.readthedocs.io/en/latest/transient/00userguide/howtos/howto_fluctuating_head_boundary.html) for why the midpoint of a time interval is used. This only influences the time scale, not the amplitudes._

In [ ]:
tsandhstar_tidal = tlt.to_tsandhstar(obs_dh[loc_river])
print(f"Tidal situation: {len(tsandhstar_tidal)} steps")

## 2. Geometry and initial parameters
Set initial parameters based on geological data. Use consistent units for length `[m]` and time `[days]`.

### 2.1 Parameters from the Excel file

In [ ]:
excel_tab  = 'initial'   # Tab name: change to 'calibrated' etc. when needed
excel_path = project.parameters_file(transect_name)

inhom_params, naq = tlt.read_model_params(excel_path, tab_name=excel_tab)

tlt.print_model_params(inhom_params)

### 2.2 Create the models (steady state + transient)
The models `mlss` and `mltr` are created below, for the steady and the transient behaviour. `naq` is the number of aquifers.

<div class="alert alert-info">
    <strong>Note:</strong> In this example the river extends to -infinity. If the river has a finite width, you can model a water divide in the center of the river. This can be accomplished in several ways:
    <ol>
        <li> by adding an impermeable wall (water divide) in the middle of the river</li>
        <li> by adding an extra inhomogeneity ranging from -infinity to the left embankment of the river</li>
    </ol>
    Method 1 is shown below.
</div>

_The models are created explicitly here. Once Timflow can save and load models ([PR #171](https://github.com/timflow-org/timflow/pull/171)), this notebook can be split into separate notebooks._

In [ ]:
# NOTE: The models are created explicitly in this notebook. Once Timflow can save and load models
# (JSON, see https://github.com/timflow-org/timflow/pull/171), this notebook can be split into
# separate notebooks for calibration, analysis and the storm event, that load the calibrated model.

hstar_river = obs_ref[loc_river].mean()    # Average head in river
hstar_polder = obs_ref[loc_farfield].mean() # Farfield reference

mlss = tf.steady.ModelXsection(naq=naq)

riv_ss = tf.steady.XsectionMaq(
    mlss,
    x1=-np.inf,  # river extends to -infinity
    x2=-L_foreland,
    kaq=inhom_params["river"]["kaq"],
    z=inhom_params["river"]["z"],
    c=inhom_params["river"]["c"],
    topboundary="semi",
    hstar=hstar_river,
    name="river",
)
fore_ss = tf.steady.XsectionMaq(
    mlss,
    x1=-L_foreland,
    x2=0,
    kaq=inhom_params["foreland"]["kaq"],
    z=inhom_params["foreland"]["z"],
    c=inhom_params["foreland"]["c"],
    topboundary="semi",
    hstar=hstar_river,  # This may not be correct for daily situation
    name="foreland",
)
hinter_ss = tf.steady.XsectionMaq(
    mlss,
    x1=0,
    x2=np.inf,
    kaq=inhom_params["hinterland"]["kaq"],
    z=inhom_params["hinterland"]["z"],
    c=inhom_params["hinterland"]["c"],
    topboundary="semi",
    hstar=hstar_polder,
    name="hinterland",
)

x_rivercenter = - L_foreland - L_river / 2
boundary_noflow_ss = tf.steady.LeakyWall1D(
    mlss, x_rivercenter, res=np.inf, layers=list(range(naq)), label="no flow boundary"
)

mlss.solve(silent=True)

Below, the transient model is defined. We deliberately do not add the steady-state model to the transient model when creating it, because we want to calculate response factors separately for the steady and the transient behaviour. The calibration takes both the steady and the transient part into account.

In [ ]:
mltr = tf.transient.ModelXsection(naq=naq, tmin=1e-5, tmax=1e2)

riv_tr = tf.transient.XsectionMaq(
    mltr,
    x1=-np.inf,  # river extends to -infinity
    x2= -L_foreland,
    kaq=inhom_params["river"]["kaq"],
    z=inhom_params["river"]["z"],
    Saq=inhom_params["river"]["Saq"],
    Sll=inhom_params["river"]["Sll"],
    c=inhom_params["river"]["c"],
    leffaq=inhom_params["river"]["leffaq"],
    leffll=inhom_params["river"]["leffll"],
    topboundary="semi",
    tsandhstar=tsandhstar_tidal,
    name="river",
)
fore_tr = tf.transient.XsectionMaq(
    mltr,
    x1=-L_foreland,
    x2=0,
    kaq=inhom_params["foreland"]["kaq"],
    z=inhom_params["foreland"]["z"],
    Saq=inhom_params["foreland"]["Saq"],
    Sll=inhom_params["foreland"]["Sll"],
    c=inhom_params["foreland"]["c"],
    leffaq=inhom_params["foreland"]["leffaq"],
    leffll=inhom_params["foreland"]["leffll"],
    topboundary="semi",
    tsandhstar=None,  # In daily situation, the foreland is dry
    name="foreland",
)
hinter_tr = tf.transient.XsectionMaq(
    mltr,
    x1=0,
    x2=np.inf,
    kaq=inhom_params["hinterland"]["kaq"],
    z=inhom_params["hinterland"]["z"],
    Saq=inhom_params["hinterland"]["Saq"],
    Sll=inhom_params["hinterland"]["Sll"],
    c=inhom_params["hinterland"]["c"],
    leffaq=inhom_params["hinterland"]["leffaq"],
    leffll=inhom_params["hinterland"]["leffll"],
    topboundary="semi",
    name="hinterland",
)

x_rivercenter = -L_foreland - L_river / 2
boundary_noflow_tr = tf.transient.LeakyWall1D(mltr, x_rivercenter, res=np.inf, layers=list(range(naq)), label="no flow boundary")

mltr.solve(silent=True)

### 2.3 Calibration object
The calibration uses the piezometers of this transect with `use_calibration` = 1 in `piezometers.xlsx` (the default). Filters with `use_calibration` = 0 are only shown in the figures, not used in the calibration, e.g. a filter that is not in an aquifer of the model; the reason is in the column `remark`. Piezometers without observations in the model period are not used either. The cell below lists per piezometer whether it is used.

In [ ]:
cal = tf.Calibrate(steady_model=mlss, transient_model=mltr)

# Piezometers with use_calibration = 0 in piezometers.xlsx are only shown, not used in the calibration
if "use_calibration" in obslocs.columns:
    use = obslocs["use_calibration"] != 0
else:
    use = pd.Series(True, index=obslocs.index)
for pb, row in obslocs[~use].iterrows():
    reason = row["remark"] if pd.notna(row.get("remark")) else "no remark in piezometers.xlsx"
    print(f"{pb}: not used in the calibration (use_calibration = 0: {reason})")

for pb, row in obslocs[use].iterrows():
    if pb not in obs_ref.columns:
        print(f"{pb}: not used in the calibration (no observations in the model period)")
        continue

    hobs = obs_ref[pb].dropna()
    cal.add_head_time_series(
        pb,
        x=row["x_crosssection"],
        y=0,
        layer=int(row["modellayer"]),
        t=hobs.index.to_numpy(),
        h=hobs.values,
    )
    print(f"{pb}: used in the calibration")

### 2.4 Cross-section of the geohydrological model
- model layout with layer numbers
- model layout with initial parameters

In [ ]:
# Some plotting parameters for all xsection plots
xsec_xlim = (-L_foreland - L_river / 2 - 10, 250)
xsec_zlim = (None, 8)
xsec_figsize = (10, 7)

# Response factors along x (summary figure): number of points over the x range of the cross-section figures
n_x_profile = 50
x_profile = np.linspace(*xsec_xlim, n_x_profile)

In [ ]:
# Plot the model layout
f, ax = tlt.plot_xsection(
    mltr,
    model_steady=mlss,
    calibrate=cal,
    xlim=xsec_xlim,
    zlim=xsec_zlim,
    figsize=xsec_figsize,
    axis_units=units,
    labels=True,
    params=False,
)
tlt.finalize_and_save(
    f, savefolder, location_name, modelrun="Model layout", tag="xsection", file_label=transect_name
)

# Plot the model layout with parameters and inhomogeneity names
f, ax = tlt.plot_xsection(
    mltr,
    model_steady=mlss,
    calibrate=cal,
    xlim=xsec_xlim,
    zlim=xsec_zlim,
    figsize=xsec_figsize,
    axis_units=units,
)
tlt.finalize_and_save(
    f, savefolder, location_name, modelrun="Run 0 Initial parameters", tag="xsection", file_label=transect_name
)

## 3. Calibration
### 3.1 Tidal fluctuation before calibration
Modelled versus observed heads at the observation wells.

In [ ]:
modelrun = "Run 1 tidal"

f, axes = tlt.plot_transient_results(cal, units=units)
tlt.finalize_and_save(f, savefolder, location_name, modelrun, tag="headT", file_label=transect_name)

### 3.2 Calibration

Choose the parameters to calibrate. The example below is the set used for the Papendrecht transects, for a scheme
with two aquifers: the resistances of both aquitards (below the river, and below foreland and hinterland together),
the conductivity of the upper aquifer, the storage of the deep aquifer and its loading efficiency below river and
foreland. Fewer parameters gave a clearly worse fit there (RMSE about 0.16 m instead of 0.10 m), mainly because the
shallow piezometers need the top aquitard and the upper aquifer. Adapt the set to your scheme: look at the fit and
at the correlations in the report, and add or remove parameters one at a time.

`cal.set_aquifer_parameter(name, layers, initial, pmin, pmax, log_scale, inhoms, model)`:
- `name`: `kaq`, `c`, `Saq`, `Sll` or `kzoverkh`
- `layers`: the layer(s); consecutive layers get one value. `c` of layer *i* is the resistance of the aquitard above aquifer *i*.
- `inhoms`: the inhomogeneities that share one value (`river`, `foreland`, `hinterland`)
- `initial`: start value; left out = the value in the geohydrological scheme
- `log_scale=True` is recommended for `kaq`, `c`, `Saq` and `Sll`
- `model`: `both` (default), `steady` or `transient` (storage only matters in the transient model)

The loading efficiency (`leffaq`, `leffll`) is not an aquifer parameter in Timflow: it is calibrated on the array of
the transient inhomogeneity. To share one value between river and foreland, the foreland is added as an extra target
(an internal function, see [Timflow issue #186](https://github.com/timflow-org/timflow/issues/186)).

In [ ]:
# Calibration parameters of the Papendrecht transects (two aquifers); start values from the geohydrological scheme
cal.set_aquifer_parameter("c", layers=0, pmin=0.01, pmax=50, log_scale=True, inhoms=["river"])
cal.set_aquifer_parameter("c", layers=0, pmin=50, pmax=1000, log_scale=True, inhoms=["foreland", "hinterland"])
cal.set_aquifer_parameter("kaq", layers=0, pmin=0.01, pmax=20, log_scale=True, inhoms=["foreland", "hinterland"])
cal.set_aquifer_parameter("c", layers=1, pmin=2, pmax=500, log_scale=True, inhoms=["river"])
cal.set_aquifer_parameter("c", layers=1, pmin=500, pmax=5000, log_scale=True, inhoms=["foreland", "hinterland"])
cal.set_aquifer_parameter(
    "Saq", layers=1, pmin=1e-6, pmax=1e-2, log_scale=True, inhoms=["river", "foreland", "hinterland"], model="transient"
)

# Loading efficiency of the deep aquifer, one value for river and foreland (see the note above)
cal.set_parameter_by_reference("leffaq", riv_tr.leffaq[1:2], initial=None, pmin=0.0, pmax=1.0)
cal._parameters["leffaq"].add_target(fore_tr.leffaq[1:2], slice(None))

In [ ]:
# Calibrate with Levenberg-Marquardt (lmfit), which also reports the correlations between the parameters
cal.lmfit()
rmse = cal.rmse()  # store now: later cells change the models
print(f"RMSE: {rmse:.5f}")

The RMSE (root mean square error) is a measure of the accuracy of the fit. An RMSE of 0.01 m means an average difference of 1 cm between modelled and measured hydraulic heads.

In [ ]:
cal.result

In [ ]:
modelrun = "Run 2 calibrated"

f, ax = tlt.plot_xsection(
    mltr,
    model_steady=mlss,
    calibrate=cal,
    xlim=xsec_xlim,
    zlim=xsec_zlim,
    figsize=xsec_figsize,
    axis_units=units,
)
tlt.finalize_and_save(
    f, savefolder, location_name, modelrun=modelrun, tag="xsection", file_label=transect_name
)

Plot modeled vs observed heads after calibration.

In [ ]:
f, axes = tlt.plot_transient_results(cal, units=units)
tlt.finalize_and_save(f, savefolder, location_name, modelrun=modelrun, tag="headT", file_label=transect_name)

Show parameters dataframe

In [ ]:
cal.parameters.iloc[:, :-2]

Check if parameters are correctly linked: `kaq`, `Saq` in all inhoms, `c`
separate in each section.

In [ ]:
mlss.aquifer_summary().style.format(na_rep="-", precision=2)

In [ ]:
(
    mltr.aquifer_summary()
    .style.format(na_rep="-", precision=2)
    .format("{:.2e}", subset=["S_s"])
)

## 4. Analysis of the calibrated model

**Response factors.** A response factor tells which part of a change of the river level reaches the groundwater at a
location $x$ in an aquifer:

$$r(x) = \frac{\Delta h(x)}{\Delta h_{\text{river}}}$$

with $\Delta$ the change relative to a reference. $r = 1$ means the full change of the river level, $r = 0$ no
response. Response factors decrease with the distance from the river (see the leakage factors in 4.1) and are usually
smaller in the upper aquifer below the blanket layer than in the deeper aquifer. In levee assessments they are used to
estimate the head during high water from the measured or modeled situation:
$h(x) = h_{\text{ref}}(x) + r(x)\,\left(h_{\text{river}} - h_{\text{river,ref}}\right)$.

The response depends on how fast the river level changes: storage in the aquifers and aquitards damps and delays fast
changes. Therefore three response factors are calculated, for three time scales:

| Response factor | Change of the river level | How it is calculated | Columns |
|---|---|---|---|
| steady (4.5) | long-term average, relative to the far-field reference | steady model: head relative to the far field divided by the river level relative to the far field; observed: the same with the average measured heads | `rf_steady_mod`, `rf_steady_obs` |
| tidal (4.2) | tidal fluctuation (period about 12.5 hours) | per tidal peak: peak of the head divided by the peak of the river level (both relative to their average), then the median over all peaks; also the time lag | `rf_tidal_mod`, `rf_tidal_obs`, `lag_tidal_..._hours` |
| design hydrograph, MHW (6.3.1) | storm event of several days | peak of the head divided by the peak of the river level: $\max h(t) / \max h_{\text{river}}(t)$ | `rf_mhw_mod` |

The tidal response factors are clearly smaller than the other two, and the heads lag behind the river: a tide is too
short for the groundwater to follow fully. A storm event lasts days, so its response comes closer to the steady one, but stays below it.
Observed response factors exist for the steady and tidal situation only, since no storm event was measured. How fast
the response decreases with the distance follows from the leakage factors of the calibrated model (4.1). Section 8
combines all response factors in one table.

### 4.1 Leakage factors and flow resistances

**Leakage factor.** In a semi-confined aquifer with transmissivity $kD$ [m²/d] below an aquitard with resistance $c$ [d],
a change of the head at a boundary (here: the river) decays with the distance $x$ from that boundary as
$e^{-x/\lambda}$, with the leakage factor

$$\lambda = \sqrt{kD \cdot c} \quad \text{[m]}$$

At a distance $\lambda$ about 63% of the change has gone, at $3\lambda$ about 95%. A large leakage factor (a well-conducting
aquifer below a tight aquitard) means that the river level is felt far into the hinterland.

With more than one aquifer, each zone has several leakage factors (the eigenvalues of the system of aquifers). Timflow
gives them per zone in `lab`, sorted from large to small; the largest (`lab[0]`) dominates the response at larger
distances and is used here. These values are taken from the steady model `mlss`: in the transient model `lab` is not a
leakage factor per layer, see [Timflow issue 187](https://github.com/timflow-org/timflow/issues/187). Below, the leakage
factor of the river zone (riverside) and of the hinterland (landside) are printed.

**Flow resistances W1 and W3.** The *Technisch Rapport Waterspanningen bij Dijken* (TRWD, TAW 2004, appendix 4,
model 4A, p. b4-5 and b4-6) schematizes the cross-section in three zones on a sand aquifer with transmissivity $kD$:
the foreland (1) with length $L_1$, the levee (2) with width $L_2$ and the hinterland (3) with length $L_3$. In the
TRWD, $\varphi_0$ is the river level, $\varphi_3$ the polder level (p.p.), and $\varphi_1$ and $\varphi_2$ are the heads
in the aquifer at the outer and the inner toe. Each zone has a flow resistance [d/m]:

$$W_1 = \frac{\lambda_1}{kD} \tanh \frac{L_1}{\lambda_1} \qquad
W_2 = \frac{L_2}{kD} \qquad
W_3 = \frac{\lambda_3}{kD} \tanh \frac{L_3}{\lambda_3}$$

with $\lambda_i = \sqrt{kD\, c_i}$ the leakage factor and $c_i = d_i / k_{vi}$ the resistance of the covering layer in
the foreland ($i = 1$) and the hinterland ($i = 3$). The total resistance is $\Sigma W = W_1 + W_2 + W_3$. The head at
the inner toe follows from (p. b4-5):

$$\varphi_2 = \varphi_3 + (\varphi_0 - \varphi_3) \frac{W_3}{\Sigma W}$$

For a long foreland and hinterland ($L_1/\lambda_1$ and $L_3/\lambda_3$ > 1.8 à 2), $W_1 \approx \lambda_1 / kD$ and
$W_3 \approx \lambda_3 / kD$, and the head in the hinterland is (p. b4-6):

$$\varphi(x) = \varphi_3 + (\varphi_2 - \varphi_3)\, e^{(a - x)/\lambda_3}$$

with $x$ measured from the centre of the levee, so the toes are at $x = -a$ and $x = a$ ($L_2 = 2a$).

In these models the levee is not a separate zone: foreland and hinterland meet at the inner toe, so $L_2 = 0$, hence
$W_2 = 0$ and $a = 0$, and $x = 0$ at the inner toe, as in the toolkit. The far-field head of the model plays the role of
the polder level $\varphi_3$. Dividing by $\varphi_0 - \varphi_3$ gives the response factor (section 4) at the inner
toe (the hinge point, *kantelpunt*) and in the hinterland:

$$\frac{\varphi_2 - \varphi_3}{\varphi_0 - \varphi_3} = \frac{W_3}{W_1 + W_3} \qquad
\frac{\varphi(x) - \varphi_3}{\varphi_0 - \varphi_3} = \frac{W_3}{W_1 + W_3}\, e^{-x/\lambda_3}$$

The leakage factors and flow resistances are calculated from the calibrated model, not fitted to the response
factors: the model itself gives the response factors along the whole cross-section (section 8).

**River zone: model and TRWD differ.** In the TRWD (figure b4.4) the river fully penetrates the aquifer: the river
level $\varphi_0$ is imposed directly on the sand at the outer end of the foreland, $x = -(a + L_1)$. In our models
the river lies on top of the aquifers: below the river there is a river zone with its own aquitard (the riverbed), so
the river level reaches the aquifer only through the resistance of the riverbed. The TRWD has no counterpart for this
zone. Therefore:

- the leakage factor *riverside* printed below is that of the river zone (`lab[0]` of the zone *river*): it describes
  how the river level enters the aquifer through the riverbed;
- $W_1$ is calculated for the foreland only, as in the TRWD: `tlt.calculate_flow_resistance` uses the leakage factor of
  the zone *foreland* and its length $L_1$ (`width_foreland` in the metadata). The resistance of the riverbed is not
  part of $W_1$, so $W_3 / (W_1 + W_3)$ overestimates the response at the inner toe if the riverbed resistance matters.

$W_3$ is calculated for the zone *hinterland* ($L_3 \to \infty$, so $W_3 = \lambda_3 / kD$). Both use $kD$ of the
deepest aquifer (`layer=-1`).

In [ ]:
leakagefactor_riverside = mlss.aq.inhomdict["river"].lab[0]
print(f"Leakagefactor riverside: {leakagefactor_riverside:5.0f} m")

leakagefactor_landside = mlss.aq.inhomdict["hinterland"].lab[0]
print(f"Leakagefactor landside:  {leakagefactor_landside:5.0f} m")

print(f"Ratio between leakagefactors: {leakagefactor_riverside / leakagefactor_landside:5.2f}")


flow_resistance_foreland = tlt.calculate_flow_resistance(
    mlss, "foreland", layer=-1
)
print(f"Flow resistance foreland W1: {flow_resistance_foreland:5.3f} d/m")

flow_resistance_hinterland = tlt.calculate_flow_resistance(
    mlss, "hinterland", layer=-1
)
print(f"Flow resistance hinterland W3: {flow_resistance_hinterland:5.3f} d/m")

print(
    f"Ratio W1/W3: {flow_resistance_foreland / flow_resistance_hinterland :5.2f}"
)

### 4.2 Tidal response factors

The transient model does not contain the steady part, so its heads are directly the response to the tidal fluctuation of the river; the tidal response factors are calculated from them (see the table at the start of section 4).

In [ ]:
modelrun = "Run 2 calibrated"
t_tidal = obs_dh.index.to_numpy()
h_river_tidal = obs_dh[loc_river]

# Heads relative to their average per piezometer: modeled (transient part only) and observed
h_mod_tidal = tlt.select_obslocs(
    tlt.calculate_head_t(mltr, obslocs["x_crosssection"].unique(), t_tidal), obslocs
).droplevel("layer", axis=1)
h_obs_tidal = obs_dh[[p for p in obslocs.index if p in obs_dh.columns]]

# Response factor and time lag per tidal peak, then the median per piezometer
peaks_tidal = {
    "mod": tlt.calculate_rf_peaks(h_mod_tidal, h_river_tidal),
    "obs": tlt.calculate_rf_peaks(h_obs_tidal, h_river_tidal),
}
df_rf_tidal = tlt.summarize_rf_tidal(peaks_tidal, obslocs)

# Modeled tidal response factor along x (with the tidal boundary condition, so before the storm event)
rf_profile_tidal = tlt.calculate_rf_tidal_x(mltr, x_profile, t_tidal, h_river_tidal)

df_rf_tidal

Response factor and time lag per tidal peak, for each piezometer (modeled and observed).

In [ ]:
for obsloc, row in obslocs.iterrows():
    f, axes = tlt.plot_rf_peaks(
        obsloc,
        h_river_tidal,
        series={"mod": h_mod_tidal[obsloc], "obs": h_obs_tidal.get(obsloc)},
        peaks=peaks_tidal,
        title=f"{obsloc} (x={row['x_crosssection']}, layer={int(row['modellayer'])})",
        units=units,
    )
    tlt.finalize_and_save(f, savefolder, transect_name, modelrun=modelrun, tag=f"peak_response_{obsloc}", file_label=transect_name)

### 4.3 Hysteresis
Piezometer head against river level over the tidal cycles, modeled and observed. Because the piezometer lags behind the river, each tide gives a loop: an ellipse for a purely sinusoidal tide. The real tide is not a pure sine (unequal daily tides, faster rise than fall), so the loops differ from tide to tide. The width of the loops shows the time lag, the slope the response factor.

In [ ]:
# Piezometer with the largest modeled tidal response (section 4.2), or choose one, e.g. obsloc = "GMW000000112214_2"
obsloc = df_rf_tidal["rf_tidal_mod"].idxmax()
xloc = obslocs.loc[obsloc, "x_crosssection"]
modellayer = int(obslocs.loc[obsloc, "modellayer"])

# Leave out the first day: the transient model starts at rest and needs about a day to follow the tide
t_spinup = 1.0  # [days]
h_river = obs_dh[loc_river].dropna()
h_river = h_river[h_river.index >= t_spinup]
h_mod = pd.Series(mltr.head(xloc, 0.0, h_river.index.to_numpy())[modellayer], index=h_river.index, name="modeled")

ax = tlt.plot_hysteresis(h_river, h_mod, units=units)
if obsloc in obs_dh.columns:
    # Observed: the river level interpolated to the times of the piezometer measurements
    h_obs = obs_dh[obsloc].dropna()
    h_obs = h_obs[h_obs.index >= t_spinup].rename("observed")
    h_river_at_obs = pd.Series(np.interp(h_obs.index, h_river.index, h_river), index=h_obs.index, name=h_river.name)
    tlt.plot_hysteresis(h_river_at_obs, h_obs, ax=ax, units=units)
ax.set_ylabel(f"{obsloc} [{units['head_rel']}]")
ax.set_title(f"{obsloc} (x={xloc}, layer={modellayer})", loc="right")
ax.legend(loc=(0, 1), frameon=False, ncol=2, fontsize="x-small")
tlt.finalize_and_save(ax.figure, savefolder, location_name, modelrun=modelrun, tag="hysteresis", file_label=transect_name)

df_rf_tidal.loc[[obsloc]]

### 4.4 Steady state
The steady state is in the model `mlss`. Heads along x:

In [ ]:
modelrun = "Calibrated steady"

f, axes = tlt.plot_steady_results(cal,
                                  x1=- L_foreland - 5,
                                  x2=200,
                                  sharey=True,
                                  units=units
                                  )
tlt.finalize_and_save(f, savefolder, location_name, modelrun, tag="headX", file_label=transect_name)

### 4.5 Steady-state response factors

In [ ]:
df_rf_steady = tlt.calculate_rf_steady(
    mlss, obslocs, obs_avg_farfieldref[loc_river], h_obs=obs_avg_farfieldref
)
rf_profile_steady = tlt.calculate_rf_steady_x(mlss, x_profile, obs_avg_farfieldref[loc_river])  # along x

df_rf_steady

## 5. Save the calibrated parameters
If you are happy with the model, save the calibrated parameters as a new tab in the parameter file. If the tab name already exists, a timestamp is added. The tab can be read again with `read_model_params`, e.g. as the start of a next calibration.

_Saving and loading complete models (including the geometry) is being developed in Timflow, see [PR #171](https://github.com/timflow-org/timflow/pull/171)._

In [ ]:
mp_calibrated = tlt.write_model_params(mltr, excel_path, tab_name="calibrated")

In [ ]:
modelrun = "Run 50 accepted model"

# Plot the model layout with parameters and inhomogeneity names
f, ax = tlt.plot_xsection(
    mltr,
    model_steady=mlss,
    calibrate=cal,
    xlim=xsec_xlim,
    zlim=xsec_zlim,
    figsize=xsec_figsize,
    axis_units=units,
)
tlt.finalize_and_save(
    f, savefolder, location_name, modelrun=modelrun, tag="xsection", file_label=transect_name
)

## 6. Storm event ('waterstandsverloop')
Use the calibrated model to compute the groundwater heads during a storm event. This part is optional and needs a storm event file (`loc_storm_event` in the metadata).

### 6.1 Load the storm event
The river is assumed to be at its average level, so the changes compared to the average river level are used, starting at t = 0.

In [ ]:
fname_storm_event = project.storm_event_file(loc_storm_event)
df_storm_event_offset = tlt.read_storm_event(
    fname_storm_event, h_river_avg=obs_ref[loc_river].mean()
)
f, axes = tlt.plot_storm_event(df_storm_event_offset, units=units)

tsandhstar_storm_event = tlt.to_tsandhstar(df_storm_event_offset["h_river_rel"])
print(f"storm event: {tsandhstar_storm_event}")

### 6.2 Assign new boundary condition

In the storm event, the foreland is wet. So here we assign the `tsandhstar` to the foreland as well.

In [ ]:
mltr.aq.inhomdict["river"].tsandhstar = tsandhstar_storm_event
mltr.aq.inhomdict["foreland"].tsandhstar = tsandhstar_storm_event
mltr.solve(silent=True)

### 6.3 Plot head along T at multiple X locations

Plot of heads (versus time) at given x-locations. Here we use the locations of the observation points (`xlocs=obslocs["x_crosssection"].unique()`), but you could as well use specific locations of interest (`xlocs = np.array([0, 10, 100])`).

In [ ]:
# Select the x-locations to plot heads for:
xlocs = obslocs["x_crosssection"].unique()
# xlocs = np.array([0, 10, 100])

modelrun = "Run 51 storm event"

# Total head (steady + transient) [m NAP] over time, all layers
t = df_storm_event_offset.index.to_numpy()
h_t = tlt.calculate_head_t(mltr, xlocs, t, model_steady=mlss)

f, axes = tlt.plot_head_t(h_t, river=df_storm_event_offset["h_river"], units=units)
tlt.finalize_and_save(f, savefolder_final, location_name, modelrun=modelrun, tag="headTriv", file_label=transect_name)

#### 6.3.1 Response factor for the design hydrograph (MHW)
MHW stands for the design situation: the design hydrograph (*waterstandsverloop*) with the design high water level (*maatgevende hoogwaterstand*) as its peak. The response factor is the peak of the head divided by the peak of the river level (both relative to the average river level): max(h(t)) / max(h_river(t)). The head reaches its peak later than the river; the figure shows the head divided by the peak of the river level, with a marker at the peak of the head.

In [ ]:
modelrun = "Run 51 storm event"
h_river = df_storm_event_offset["h_river_rel"]

# Transient head (without steady part) per piezometer in its own model layer
h_tr_t = tlt.calculate_head_t(mltr, obslocs["x_crosssection"].unique(), h_river.index.to_numpy())
h_tr_t = tlt.select_obslocs(h_tr_t, obslocs)
df_rf_mhw = tlt.calculate_rf_mhw(h_tr_t, h_river, obslocs)
rf_profile_mhw = tlt.calculate_rf_mhw_x(mltr, x_profile, h_river)  # along x

# Head divided by the peak of the river level; its maximum is the response factor
rf_t = h_tr_t / h_river.max()
f, axes = tlt.plot_head_t(rf_t, markers=df_rf_mhw[["t_mhw_mod", "rf_mhw_mod"]], ylabel="response factor [-]", units=units)
tlt.finalize_and_save(f, savefolder_final, location_name, modelrun=modelrun, tag="responsefactor_peak", file_label=transect_name)

df_rf_mhw

### 6.4 Plot head along X at multiple times

The maximum of the presented time steps is calculated.

In [ ]:
t = np.arange(0.1, 6.5, 0.5)
x = np.linspace(-L_river - L_foreland, 300, 501)

# Heads along x: steady (layer, x) + transient (layer, time, x) = total [m NAP]
h_ss = mlss.headalongline(x, np.zeros_like(x))
h_tr = mltr.headalongline(x, np.zeros_like(x), t)
h_total = h_ss[:, np.newaxis] + h_tr

curves = {f"t = {ti:.1f} d": h_total[:, i] for i, ti in enumerate(t)}
curves["max"] = h_total.max(axis=1)
f, axes = tlt.plot_head_x(
    x, curves, styles={"max": {"color": "k", "linestyle": ":"}}, cmap="Spectral", units=units
)
tlt.finalize_and_save(f, savefolder_final, location_name, modelrun=modelrun, tag="headX2", file_label=transect_name)

## 7. Maximum heads during the storm in m NAP
The **maximum head during storm conditions in m NAP** is the maximum over time of the total head (steady state + transient response to the storm event) calculated along x in section 6.4.

In [ ]:
modelrun = "Run 52 max heads storm"
h_max = h_total.max(axis=1)

df_heads = pd.DataFrame(h_max.T, index=x)
df_heads.index.name = f"x [{units['x']}]"

f, ax = plt.subplots(figsize=(10, 5))
df_heads.plot(ylabel=f"Head [{units['head']}]", grid=True, ax=ax)
ax.set_title("Maximum heads per modellayer during a storm", loc="right")
tlt.finalize_and_save(f, savefolder, location_name, modelrun=modelrun, tag="maxheads", file_label=transect_name)

### 7.1 Total head over time at the piezometer locations

In [ ]:
# Select the x-locations to plot heads for:
xlocs = obslocs["x_crosssection"].unique()
# xlocs = np.array([0, 10, 100])

modelrun = "Run 61 storm event total head"

# Total head (steady + transient) [m NAP] over time, all layers
t = df_storm_event_offset.index.to_numpy()
h_t = tlt.calculate_head_t(mltr, xlocs, t, model_steady=mlss)

f, axes = tlt.plot_head_t(h_t, river=df_storm_event_offset["h_river"], units=units)
tlt.finalize_and_save(f, savefolder, location_name, modelrun=modelrun, tag="headTriv", file_label=transect_name)

## 8. Summary
All response factors in one table (one row per piezometer) and one Excel file:
- steady state (`rf_steady_mod`, `rf_steady_obs`)
- design hydrograph (MHW, `rf_mhw_mod`)
- tidal: median over all tidal peaks (`rf_tidal_mod`, `rf_tidal_obs`) and the time lag in hours

The response factors along x (the lines in the figure below, `n_x_profile` points over the x range of the
cross-section figures, all layers) are saved as `responsefactors_x.xlsx`. These files and `model_summary.xlsx` in the
'results_final' folder of the transect are combined over all transects in notebook 60.

In [ ]:
# Model summary of this transect (one row), combined over all transects in notebook 60
model_summary = pd.DataFrame([{
    "transect": transect_name,
    "parameter_tab": mp_calibrated.attrs.get("tab_name"),
    "rmse_m": rmse,
    "leakage_factor_riverside_m": leakagefactor_riverside,
    "leakage_factor_landside_m": leakagefactor_landside,
    "flow_resistance_W1_d_per_m": flow_resistance_foreland,
    "flow_resistance_W3_d_per_m": flow_resistance_hinterland,
}]).set_index("transect")
tlt.export_to_excel(model_summary, savefolder_final / "model_summary.xlsx")

model_summary.T

In [ ]:
# Order: steady, design hydrograph (MHW), tidal (from slow to fast changes of the river level)
df_rf = tlt.combine_rf(obslocs, df_rf_steady, df_rf_mhw, df_rf_tidal)

rf_columns = [c for c in df_rf.columns if c.startswith("rf_")]
lag_columns = [c for c in df_rf.columns if c.startswith("lag_")]
tlt.export_to_excel(
    df_rf,
    savefolder_final / "responsefactors.xlsx",
    number_formats={**{c: "0.0%" for c in rf_columns}, **{c: "0.0" for c in lag_columns}},
)

df_rf.style.format({c: "{:.1%}" for c in rf_columns}, precision=2, na_rep="-")

Response factors along the cross-section: modeled as lines along x (`n_x_profile` points), at the piezometers as markers (open: observed).

In [ ]:
rf_profiles = {"rf_steady_mod": rf_profile_steady, "rf_mhw_mod": rf_profile_mhw, "rf_tidal_mod": rf_profile_tidal}
f, axes = tlt.plot_rf_x(df_rf, profiles=rf_profiles, units=units)
tlt.finalize_and_save(f, savefolder_final, location_name, modelrun="Summary", tag="responsefactors_x", file_label=transect_name)

# Response factors along x of all layers, for notebook 60
df_rf_x = tlt.combine_rf_profiles(rf_profiles)
tlt.export_to_excel(
    df_rf_x, savefolder_final / "responsefactors_x.xlsx", number_formats={c: "0.0%" for c in rf_profiles}, index=False
)

---

<!-- navigation -->
[<kbd>← 51 Prepare model inputs</kbd>](51%20Prepare%20model%20inputs.ipynb) &nbsp; [<kbd>Overview</kbd>](README.md) &nbsp; [<kbd>60 Summary over transects →</kbd>](60%20Summary%20over%20transects.ipynb)